In [ ]:
import os
import math
import yaml
from typing import Callable

import numpy as np
import sys
import carla
import random
import matplotlib.pyplot as plt


# Append to access src
sys.path.append('../')

from src.client.core.carlautils import get_ordered_spawn_points
from src.client.core.carlautils import create_transform_from_coordinates, randomize_transform, vector3d_to_dict, rotation_to_dict


# Functions or tools to get information for scenario config creation

### Access CARLA
- Carla (local or container) must be already running.

In [ ]:
client = carla.Client("localhost", 2000)
world = client.get_world()
blueprint_library = world.get_blueprint_library()
weather = world.get_weather()
sim_map = world.get_map()


In [ ]:
# See all available maps
client.get_available_maps()

In [ ]:
client.load_world("Town05")
world = client.get_world()
sim_map = world.get_map()


### Get Spectator location and rotation for Config

In [ ]:
spectator = world.get_spectator()
transform = spectator.get_transform()
location = transform.location
rotation = transform.rotation

location_dict = {
    "x": round(location.x, 2),
    "y": round(location.y, 2),
    "z": round(location.z, 2),
}

rotation_dict = {
    "pitch": round(rotation.pitch, 2),
    "yaw": round(rotation.yaw, 2),
    "roll": round(rotation.roll, 2),
}

print(f'"location": {location_dict},', f'"rotation": {rotation_dict},')

### Show spawn points

In [ ]:
# viewpoints = sim_map.generate_waypoints(2.0)
# for i, w in enumerate(viewpoints):
#     world.debug.draw_string(w.transform.location, f'{i}', draw_shadow=False,
#                             color=carla.Color(r=255, g=0, b=0), life_time=120.0,
#                             persistent_lines=True)

## Plot spawn point map
- Order of spawn points is not persistent. Must be sorted.


In [ ]:

class Line:
    x = []
    y = []


class MapVisualization:
    def __init__(self, client, figsize: tuple):
        self.carla_client = client
        self.world = self.carla_client.get_world()
        self.map = self.world.get_map()
        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.ax.set_xlabel('X')
        self.ax.set_ylabel('Y')
        self.line_list = []

    def destroy(self):
        self.carla_client = None
        self.world = None
        self.map = None

    @staticmethod
    def lateral_shift(transform, shift):
        """Makes a lateral shift of the forward vector of a transform"""
        transform.rotation.yaw += 90
        return transform.location + shift * transform.get_forward_vector()

    def draw_line(self, points: list):
        x = []
        y = []
        for p in points:
            x.append(p.x)
            y.append(-p.y)
        line = Line()
        line.x = x
        line.y = y
        self.line_list.append(line)
        self.ax.plot(x, y, color='darkslategrey', markersize=2)
        return True

    def draw_spawn_points(self, spawn_points: list):
        for i, p in enumerate(spawn_points):
            x = p.location.x
            y = -p.location.y
            self.ax.text(x, y, str(i),
                         fontsize=8,
                         color='darkorange',
                         va='center',
                         ha='center',
                         weight='bold')

    def draw_roads(self):
        precision = 0.1
        topology = self.map.get_topology()
        topology = [x[0] for x in topology]
        topology = sorted(topology, key=lambda w: w.transform.location.z)
        set_waypoints = []
        for waypoint in topology:
            waypoints = [waypoint]
            nxt = waypoint.next(precision)
            if len(nxt) > 0:
                nxt = nxt[0]
                while nxt.road_id == waypoint.road_id:
                    waypoints.append(nxt)
                    nxt = nxt.next(precision)
                    if len(nxt) > 0:
                        nxt = nxt[0]
                    else:
                        break
            set_waypoints.append(waypoints)

        for waypoints in set_waypoints:
            # waypoint = waypoints[0]
            road_left_side = [self.lateral_shift(w.transform, -w.lane_width * 0.5) for w in waypoints]
            road_right_side = [self.lateral_shift(w.transform, w.lane_width * 0.5) for w in waypoints]

            # road_points = road_left_side + [x for x in reversed(road_right_side)]
            # self.add_line_strip_marker(points=road_points)

            if len(road_left_side) > 2:
                self.draw_line(points=road_left_side)
            if len(road_right_side) > 2:
                self.draw_line(points=road_right_side)


spawn_points = get_ordered_spawn_points(world=world)
viz = MapVisualization(client, figsize=(40, 40))
viz.draw_roads()
viz.draw_spawn_points(spawn_points)
viz.destroy()
plt.axis('equal')
plt.show()

# Generate Camera Grid

In [ ]:
def draw_camera_vectors(viewpoints: list[carla.Transform], life_time: float = 30.0, color: carla.Color = carla.Color(0, 0, 255)):
    for i, w in enumerate(viewpoints):

        # Draw index label
        world.debug.draw_string(
            w.location,
            f'{i}',
            draw_shadow=False,
            color=color,
            life_time=life_time,
        )

        pitch = math.radians(w.rotation.pitch)
        yaw   = math.radians(w.rotation.yaw)
        roll  = math.radians(w.rotation.roll)

        forward = carla.Vector3D(
            x = math.cos(pitch) * math.cos(yaw),
            y = math.cos(pitch) * math.sin(yaw),
            z = math.sin(pitch)
        )

        scale = 2.0
        end_location = w.location + forward * scale

        world.debug.draw_line(
            w.location,
            end_location,
            thickness=0.1,
            color=color,
            life_time=life_time,
        )


def draw_point(point: tuple[float, float, float], size: float = 0.1, life_time: float = 30.0, color: carla.Color = carla.Color(0, 0, 255)):
    world.debug.draw_point(
        carla.Location(x=point[0], y=point[1], z=point[2]),
        size=size,
        color=color,
        life_time=life_time,
    )


In [ ]:
sensors = [
    {
        "location": {'x': -32.61408615112305, 'y': -91.88381958007812, 'z': 36.214805603027344},
        "rotation": {'pitch': -48.537498474121094, 'yaw': -176.3747100830078, 'roll': -0.08140309900045395},
        "location_scaling": {'x': 1, 'y': 1, 'z': 1},
        "rotation_scaling": {'pitch': 5, 'yaw': 5, 'roll': 2.5},
        "static_camera": True,
    },
    {
        "location": {'x': -35.55815887451172, 'y': -83.69060516357422, 'z': 33.065277099609375},
        "rotation": {'pitch': -46.31218719482422, 'yaw': -162.91091918945312, 'roll': 4.082622528076172},
        "location_scaling": {'x': 1, 'y': 1, 'z': 1},
        "rotation_scaling": {'pitch': 5, 'yaw': 5, 'roll': 2.5},
        "static_camera": True,
    },
    {
        "location": {'x': -42.197532653808594, 'y': -79.1544189453125, 'z': 28.661048889160156},
        "rotation": {'pitch': -54.13483428955078, 'yaw': -133.4720458984375, 'roll': -0.1093599945306778},
        "location_scaling": {'x': 1, 'y': 1, 'z': 1},
        "rotation_scaling": {'pitch': 5, 'yaw': 5, 'roll': 2.5},
        "static_camera": True,
    },
    {
        "location": {'x': -47.394142150878906, 'y': -72.85671997070312, 'z': 34.37980270385742},
        "rotation": {'pitch': -64.40089416503906, 'yaw': -103.6722640991211, 'roll': 3.9392144680023193},
        "location_scaling": {'x': 1, 'y': 1, 'z': 1},
        "rotation_scaling": {'pitch': 5, 'yaw': 5, 'roll': 2.5},
        "static_camera": True,
    },
    {
        "location": {'x': -55.1038932800293, 'y': -71.48920440673828, 'z': 35.13174819946289},
        "rotation": {'pitch': -67.15039825439453, 'yaw': -83.8469009399414, 'roll': -0.5410407185554504},
        "location_scaling": {'x': 1, 'y': 1, 'z': 1},
        "rotation_scaling": {'pitch': 5, 'yaw': 5, 'roll': 2.5},
        "static_camera": True,
    },
    {
        "location": {'x': -62.351314544677734, 'y': -75.76834869384766, 'z': 32.066043853759766},
        "rotation": {'pitch': -65.92951202392578, 'yaw': -47.99799346923828, 'roll': -2.483473539352417},
        "location_scaling": {'x': 1, 'y': 1, 'z': 1},
        "rotation_scaling": {'pitch': 5, 'yaw': 5, 'roll': 2.5},
        "static_camera": True,
    },
    {
        "location": {'x': -68.47322082519531, 'y': -80.23929595947266, 'z': 32.12909698486328},
        "rotation": {'pitch': -67.03057098388672, 'yaw': -23.598594665527344, 'roll': 1.8467137813568115},
        "location_scaling": {'x': 1, 'y': 1, 'z': 1},
        "rotation_scaling": {'pitch': 5, 'yaw': 5, 'roll': 2.5},
        "static_camera": True,
    },
]

sensors = [create_transform_from_coordinates(sensor) for sensor in sensors]

draw_camera_vectors(sensors)


In [ ]:
def fit_sphere(points: list[tuple[float, float, float]]) -> tuple[tuple[float, float, float], float]:
    P = np.array(points)
    X, Y, Z = P[:,0], P[:,1], P[:,2]

    A = np.column_stack([2*X, 2*Y, 2*Z, np.ones(len(P))])
    b = X**2 + Y**2 + Z**2

    c, residules, rank, singval = np.linalg.lstsq(A,b)

    cx, cy, cz, d = c
    r = math.sqrt(cx**2 + cy**2 + cz**2 + d)

    return (cx, cy, cz), r


def fit_sphere_simple(points, z_offset: float = 0.0) -> tuple[np.ndarray, float]:
    P = np.array(points)
    X, Y, Z = P[:,0], P[:,1], P[:,2]

    mean = np.mean(P, axis=0)
    map_level_mean = mean

    # map_level_mean = np.zeros_like(mean)
    # map_level_mean[0:2] = mean[0:2]
    map_level_mean[2] += z_offset

    r = np.mean(np.linalg.norm(P - map_level_mean, axis=1))

    return map_level_mean, r


def forward_to_rotation(forward: np.ndarray,) -> carla.Rotation:
    fx, fy, fz = forward

    # yaw: rotation around z
    yaw = math.degrees(math.atan2(fy, fx))

    # pitch: rotation around y
    pitch = math.degrees(math.atan2(fz, math.sqrt(fx*fx + fy*fy)))

    # roll stays 0 (because all points lie on top hemisphere)
    return carla.Rotation(pitch=pitch, yaw=yaw, roll=0.0)


def parametric_sphere_transform(
    center: tuple[float, float, float] | np.ndarray,
    radius: float,
    theta: float,
    phi: float,
    radius_variation: Callable | None = None,
    location_scaling: dict[str, float] | None = None,
    rotation_scaling: dict[str, float] | None = None,
) -> carla.Transform:
    cx, cy, cz = center
    if radius_variation is None:
        r = radius
    else:
        r = radius + radius_variation(radius)

    x = cx + r * math.sin(theta) * math.cos(phi)
    y = cy + r * math.sin(theta) * math.sin(phi)
    z = cz + r * math.cos(theta)

    forward = np.array([x - cx, y - cy, z - cz], dtype=float)
    forward /= np.linalg.norm(forward)

    rotation = forward_to_rotation(-forward)

    transform = carla.Transform(carla.Location(x, y, z), rotation)
    transform = randomize_transform(
        transform, location_scaling=location_scaling, rotation_scaling=rotation_scaling,
    )
    return transform


def generate_hemisphere_transforms(
    center: tuple[float, float, float] | np.ndarray,
    radius: float,
    theta_steps: int = 5,
    phi_steps: int = 10,
    theta_offset: float = math.pi / 6,
    radius_variation: Callable | None = None,
    location_scaling: dict[str, float] | None = None,
    rotation_scaling: dict[str, float] | None = None,
) -> list[carla.Transform]:
    transforms = []
    thetas = np.linspace(theta_offset, math.pi / 2, theta_steps, endpoint=False)
    phis = np.linspace(0, 2 * math.pi, phi_steps, endpoint=False)
    for theta in thetas:
        for phi in phis:
            T = parametric_sphere_transform(center, radius, theta, phi, radius_variation=radius_variation, location_scaling=location_scaling, rotation_scaling=rotation_scaling)
            transforms.append(T)

    return transforms


In [ ]:
def filter_transforms_visible(world: carla.World, transforms, sphere_center, ignore_actors=[]):
    """
    Keep only transforms where the line from location to sphere_center
    does NOT hit any opaque object.
    """
    removed_indexes = []
    filtered = []
    if isinstance(sphere_center, tuple) or isinstance(sphere_center, list) or isinstance(sphere_center, np.ndarray):
        sphere_center = carla.Location(*sphere_center)

    for i, T in enumerate(transforms):
        start = T.location
        hit_result = world.cast_ray(start, sphere_center)

        if len(hit_result) == 0:
            filtered.append(T)
            continue
        else:
            removed_indexes.append(i)

    return filtered, removed_indexes


In [ ]:
# Convert CARLA vectors to numpy
z_offset = 0
theta_steps = 4
phi_steps = 15

points = [(v.location.x, v.location.y, v.location.z) for v in sensors]

# center, radius = fit_sphere_simple(points, z_offset)
center = np.array([location.x, location.y, location.z])
radius = 39

center_delta = np.array([-5.0, -2.0, 0.0])
center += center_delta

draw_point(center, size=0.2, color=carla.Color(255, 0, 0))

radius_variation = lambda x: 10 * (random.random() - 0.5)

location_scaling = {"x": 1, "y": 1, "z": 1}
rotation_scaling = {"pitch": 30, "yaw": 30, "roll": 10}

transforms = generate_hemisphere_transforms(
    center, radius,
    theta_steps=theta_steps,
    phi_steps=phi_steps,
    radius_variation=radius_variation,
    location_scaling=location_scaling,
    rotation_scaling=rotation_scaling,
)

draw_camera_vectors(transforms)

remove_indexes = [
    10, 11, 12, 24, 25, 26, 27, 39, 40, 41, 42, 51, 54, 55, 56, 57, 52, 58, 43
]

if len(remove_indexes) == 0:
    transforms_visible, remove_indexes = filter_transforms_visible(
        world,
        transforms,
        sphere_center=center,
    )
    print(remove_indexes)
else:
    transforms_visible = [tr for i, tr in enumerate(transforms) if i not in remove_indexes]

draw_camera_vectors(transforms_visible, color=carla.Color(0, 255, 0))


In [ ]:
print("Num. transforms:", len(transforms_visible))

In [ ]:
def format_transforms_to_sensor_defs(
    transforms: list[carla.Transform],
    location_scaling: dict[str, float],
    rotation_scaling: dict[str, float],
    static_camera: bool = True,
):
    print("sensors: [")
    for i, transform in enumerate(transforms):
        loc = vector3d_to_dict(transform.location)
        rot = rotation_to_dict(transform.rotation)
        print("  {")
        print(f"    \"location\": {loc},")
        print(f"    \"rotation\": {rot},")
        print(f"    \"location_scaling\": {location_scaling},")
        print(f"    \"rotation_scaling\": {rotation_scaling},")
        print(f"    \"static_camera\": {static_camera},")
        if i == len(transforms) - 1:
            print("  }")
        else:
            print("  },")
    print("]")



location_scaling = {"x": 1, "y": 1, "z": 1}
rotation_scaling = {"pitch": 5, "yaw": 5, "roll": 2.5}

format_transforms_to_sensor_defs(transforms_visible, location_scaling, rotation_scaling, static_camera=True)


### Spawn an actor

In [ ]:
spawned_actors = []


In [ ]:
static_transform = carla.Transform(
    location,
    rotation,
)

bp_name = [
    "static.prop.streetbarrier",
]
blueprints = []
for bp in world.get_blueprint_library():
    if any(model in bp.id for model in bp_name):
        for attr in bp:
            print(attr)
            if attr.is_modifiable:
                bp.set_attribute(attr.id, random.choice(attr.recommended_values))
        blueprints.append(bp)

actor = world.try_spawn_actor(random.choice(blueprints), static_transform)

spawned_actors.append(actor)


In [ ]:
for actor in spawned_actors:
    actor.destroy()
spawned_actors = []